# Tutorial 1: Quickstart Scientific Taste

Estimated time: 20-30 minutes

## Prerequisites
No optional dependencies required.

## Learning aims
- Primary package aim: run `validate -> plan -> run -> runs list/show` on a typed spec
- Secondary scientific aim: relate input perturbations (`a`, `b`) to output behavior (`sum`, `product`)

## Success criteria
- you can explain one observed trend in `sum` and one in `product` from the heatmaps


## Why this tutorial matters
Each step connects CLI execution to scientific reasoning.
You will generate one centralized sweep table and analyze both toy outputs directly from that table.


## Step 1: Validate and plan


In [ ]:
import io
import os
import sys
from contextlib import contextmanager, redirect_stderr, redirect_stdout
from pathlib import Path

SPEC_PATH = "tutorials/specs/model.toy.grid.json"

root = Path.cwd().resolve()
if not (root / "src").is_dir() and (root.parent / "src").is_dir():
    root = root.parent
if not (root / "src").is_dir():
    raise RuntimeError("Could not locate project root (expected src/).")

src_path = root / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

from metamodeler.cli.main import main as mm_main


@contextmanager
def in_project_root():
    previous = Path.cwd()
    os.chdir(root)
    try:
        yield
    finally:
        os.chdir(previous)


def run_mm(*args: str, check: bool = True) -> int:
    stdout_buf = io.StringIO()
    stderr_buf = io.StringIO()
    previous_argv = sys.argv[:]
    try:
        sys.argv = ["mm", *args]
        with in_project_root(), redirect_stdout(stdout_buf), redirect_stderr(stderr_buf):
            exit_code = mm_main()
    finally:
        sys.argv = previous_argv

    print("$ mm", " ".join(args))
    stdout_text = stdout_buf.getvalue().strip()
    stderr_text = stderr_buf.getvalue().strip()
    if stdout_text:
        print(stdout_text)
    if stderr_text:
        print(stderr_text)

    if check and exit_code != 0:
        raise RuntimeError(f"CLI command failed ({exit_code}): mm {' '.join(args)}")
    return exit_code


run_mm("validate", SPEC_PATH)
run_mm("plan", SPEC_PATH)


## Step 2: Execute sweep and list run IDs


In [ ]:
run_mm("run", SPEC_PATH)
run_mm("runs", "list")


## Step 3: Auto-select latest toy sweep and inspect run record


In [ ]:
import json

registry_path = root / "tmp" / "run_registry.json"


def collect_toy_sweeps() -> list[tuple[str, str, dict]]:
    if not registry_path.exists():
        return []
    registry = json.loads(registry_path.read_text())
    matches: list[tuple[str, str, dict]] = []
    for run_id, record_path in registry.items():
        candidate = Path(record_path)
        if not candidate.is_absolute():
            candidate = root / candidate
        if not candidate.exists():
            continue
        record = json.loads(candidate.read_text())
        sweep_rows_path = record.get("sweep_rows_path", "")
        if sweep_rows_path.startswith("tmp/tutorials/toy_store/sweeps/"):
            matches.append((record.get("finished_at", ""), run_id, record))
    return matches


candidates = collect_toy_sweeps()
if not candidates:
    print("No existing toy sweep found. Running one now...")
    run_mm("run", SPEC_PATH)
    candidates = collect_toy_sweeps()

if not candidates:
    raise RuntimeError("No tutorial toy sweep runs found after attempting execution.")

_, run_id, run_record = sorted(candidates)[-1]
print("Using run id:", run_id)
print("Centralized sweep CSV:", run_record["sweep_rows_path"])
run_mm("runs", "show", run_id, check=False)

sweep_rows_path = root / run_record["sweep_rows_path"]
if not sweep_rows_path.exists():
    raise RuntimeError(f"Sweep CSV does not exist: {sweep_rows_path}")


## Step 4: Visualize centralized sweep outputs as two heatmaps


In [ ]:
import matplotlib.pyplot as plt

from metamodeler.tutorials import load_toy_heatmap_grids

a_vals, b_vals, sum_grid, prod_grid = load_toy_heatmap_grids(sweep_rows_path)

fig, axes = plt.subplots(1, 2, figsize=(12, 4), constrained_layout=True)

im0 = axes[0].imshow(sum_grid, origin="lower", cmap="viridis", aspect="auto")
axes[0].set_title("sum = a + b")
axes[0].set_xlabel("b index")
axes[0].set_ylabel("a index")
axes[0].set_xticks(range(len(b_vals)), labels=[f"{v:g}" for v in b_vals])
axes[0].set_yticks(range(len(a_vals)), labels=[f"{v:g}" for v in a_vals])
fig.colorbar(im0, ax=axes[0], label="sum")

im1 = axes[1].imshow(prod_grid, origin="lower", cmap="magma", aspect="auto")
axes[1].set_title("product = a * b")
axes[1].set_xlabel("b index")
axes[1].set_ylabel("a index")
axes[1].set_xticks(range(len(b_vals)), labels=[f"{v:g}" for v in b_vals])
axes[1].set_yticks(range(len(a_vals)), labels=[f"{v:g}" for v in a_vals])
fig.colorbar(im1, ax=axes[1], label="product")

plt.show()


## Scientific checkpoint
- `sum` heatmap should increase linearly with both `a` and `b`.
- `product` heatmap should be low near zeros and increase fastest when both `a` and `b` are high.

These patterns confirm the toy model equations and show how centralized sweep tables support immediate response-surface analysis.


## Common mistakes
- Running the notebook from the wrong folder without `src/` in scope.
- Forgetting to run Step 2 before attempting sweep visualization.
- Assuming failed rows are included in heatmaps (this tutorial filters to `status == success`).
